# Sweep a hyperparameter

Four short runs instead of one long one. Each iteration is its own
`orinth.runs.start(...)`, so the rail ends up with four named entries you can
compare rather than one run whose numbers moved for reasons nobody wrote down.

Text rather than images, because the loop has to finish while you are watching
it — the pattern is the same for any model.

In [ ]:
import orinth
import polars as pl

source = max(
    (
        dataset
        for dataset in orinth.datasets.list()
        if dataset.task_type == "text_classification" and dataset.readiness.trainable
    ),
    key=lambda dataset: dataset.total_items,
)
train = orinth.datasets.load(source.id, "train", limit=4000)
valid = orinth.datasets.load(source.id, "valid", limit=1000)
LABELS = sorted(train["label"].unique().to_list())
print(source.name, "|", train.height, "train ·", valid.height, "valid ·", LABELS)

## One vectorizer, adapted once

`TextVectorization.adapt()` walks the whole corpus, so doing it inside the sweep
would spend most of the wall clock rebuilding an identical vocabulary and would
also let the vocabulary differ between arms — which makes the arms
incomparable, quietly.

In [ ]:
import numpy as np
import tensorflow as tf

VOCAB, LENGTH = 12_000, 200
INDEX = {label: position for position, label in enumerate(LABELS)}

vectorize = tf.keras.layers.TextVectorization(
    max_tokens=VOCAB, output_sequence_length=LENGTH
)
vectorize.adapt(train["text"].to_list())


def encode(frame):
    x = vectorize(tf.constant(frame["text"].to_list()))
    y = np.array([INDEX[label] for label in frame["label"].to_list()])
    return x, y


x_train, y_train = encode(train)
x_valid, y_valid = encode(valid)
print(x_train.shape, x_valid.shape)

## The sweep

`params=` is what makes a run readable a week later: the rail shows the name,
and the run file records the exact setting that produced the curve.

In [ ]:
def build(learning_rate):
    model = tf.keras.Sequential(
        [
            tf.keras.layers.Input(shape=(LENGTH,), dtype="int64"),
            tf.keras.layers.Embedding(VOCAB, 64),
            tf.keras.layers.GlobalAveragePooling1D(),
            tf.keras.layers.Dropout(0.2),
            tf.keras.layers.Dense(len(LABELS), activation="softmax"),
        ]
    )
    model.compile(
        optimizer=tf.keras.optimizers.Adam(learning_rate),
        loss="sparse_categorical_crossentropy",
        metrics=["accuracy"],
    )
    return model


RATES = [3e-4, 1e-3, 3e-3, 1e-2]
EPOCHS = 4
results = []

for rate in RATES:
    with orinth.runs.start(
        f"lr={rate:g}", params={"lr": rate, "epochs": EPOCHS}, dataset_id=source.id, tags=["sweep"]
    ) as run:
        model = build(rate)
        for epoch in range(EPOCHS):
            model.fit(x_train, y_train, epochs=1, batch_size=64, verbose=0)
            loss, accuracy = model.evaluate(x_valid, y_valid, verbose=0)
            run.log(epoch, val_loss=loss, val_accuracy=accuracy)
        results.append({"lr": rate, "val_loss": loss, "val_accuracy": accuracy})
        print(f"lr={rate:<8g} val_loss={loss:.4f} val_accuracy={accuracy:.4f}")

## Rank them

The winner here is a candidate, not a conclusion: four arms at one seed is
inside the noise on most datasets. Re-run the top two with a different seed
before you believe the ordering.

In [ ]:
pl.DataFrame(results).sort("val_accuracy", descending=True)